In [109]:
import os
import pandas as pd
from datetime import datetime
import pydicom
import matplotlib.image as mpimg
import matplotlib.pyplot as plt
import csv
import json

from utils.selection import *


# Source of RAW DICOMs
BERN_PATH='/nasdata/Berner_Daten/Other_Side_Bilddaten' 
# Project data path
ANALYSIS_PATH='/home/bule/berner_daten_analyse/data'

APP_DATA_PATH = '/home/bule/berner_daten_analyse/apps/app_manual_preselection'

-test selection pascal

In [5]:
with open(os.path.join(APP_DATA_PATH,'pascal_sequences.json')) as json_file:
    selection_pascal_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_pascal_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Pascal'  
            })
selection_pascal_df = pd.DataFrame(rows)
selection_pascal_df['Sequence_path']=selection_pascal_df.apply(lambda x: os.path.join(BERN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)

selection_pascal_df.head()

dwi_t1_df_pascal_test=pd.DataFrame(columns=['Patient_ID','Study_ID','Sequence_ID','File_Name','Date','Sequence_ID_split,','Modality'])


###### BRING THE DF IN THE CORRECT FORMAT for the 2_0_  <------------------------------- SHOULD ALSO BE DONE FOR THE SELECTICON FROMN HAKIM ETC

dwi_t1_df_pascal_test = []  # 

for _, row in selection_pascal_df.iterrows():
    for dcm_ending in os.listdir(row['Sequence_path']):        
        dwi_t1_df_pascal_test.append({
            'Patient_ID': row['Patient_ID'],
            'Study_ID': row['Study_ID'],
            'Sequence_ID': row['Sequence_ID'],
            'File_Name': dcm_ending,
            'Full_Path': os.path.join(row['Sequence_path'], dcm_ending),
            'Modality': row['Modality'].upper()
        })

dwi_t1_df_pascal_test = pd.DataFrame(dwi_t1_df_pascal_test)
dwi_t1_df_pascal_test['Date'] = dwi_t1_df_pascal_test['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
dwi_t1_df_pascal_test['Sequence_ID_split'] = dwi_t1_df_pascal_test['Sequence_ID'].apply(lambda x: split_at_anynumber_seq(x))
dwi_t1_df_pascal_test.to_csv(os.path.join(ANALYSIS_PATH, 'dwi_t1_df_pascal_test.csv'), index=False)

## Selection by people with domain knowledge

- hakim

In [ ]:
with open(os.path.join(APP_DATA_PATH,'xbzu_sequences.json')) as json_file:
    selection_hakim_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_hakim_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Hakim'  
            })
selection_hakim_df = pd.DataFrame(rows).drop_duplicates()
selection_hakim_df['Sequence_path']=selection_hakim_df.apply(lambda x: os.path.join(BERN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)
selection_hakim_df['Date'] = selection_hakim_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

# Filter to keep only dwi and t1 modalities
df_filtered = selection_hakim_df[selection_hakim_df["Modality"].isin(["dwi", "t1"])]

first_dwi = df_filtered[df_filtered["Modality"] == "dwi"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()
first_t1 = df_filtered[df_filtered["Modality"] == "t1"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()

# only get thoes with exactly 2 mods
paired_patients_hakim_ids = pd.merge(first_dwi, first_t1, on="Patient_ID", suffixes=("_dwi", "_t1"))['Patient_ID']
paired_patients_hakim=pd.concat([first_dwi,first_t1])
paired_patients_hakim=paired_patients_hakim[paired_patients_hakim["Patient_ID"].isin(paired_patients_hakim_ids)]
unpaired_patients_hakim=selection_hakim_df[~selection_hakim_df["Patient_ID"].isin(paired_patients_hakim_ids)]
paired_patients_hakim.sort_values("Patient_ID", inplace=True)
paired_patients_hakim.info()

select the first ones per diw per patient

In [ ]:
# TODO implement MERGE LOGIC for selection files -> output paths for each patient 1 dwi and 1 anatomical seq if the work is done by all  labellers..

- julian

In [ ]:
with open(os.path.join(APP_DATA_PATH,'Julian_sequences.json')) as json_file:
    selection_julian_dict = json.load(json_file)

rows = []
for patid, selection_dict in selection_julian_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Julian'  
            })
selection_julian_df = pd.DataFrame(rows).drop_duplicates()
len(selection_julian_df)
selection_julian_df['Date'] = selection_julian_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

selection_julian_df['Sequence_path']=selection_julian_df.apply(lambda x: os.path.join(BERN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)
selection_julian_df


# Filter to keep only dwi and t1 modalities
df_filtered = selection_julian_df[selection_julian_df["Modality"].isin(["dwi", "t1"])]

first_dwi = df_filtered[df_filtered["Modality"] == "dwi"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()
first_t1 = df_filtered[df_filtered["Modality"] == "t1"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()

# only get thoes with exactly 2 mods
paired_patients_julian_ids = pd.merge(first_dwi, first_t1, on="Patient_ID", suffixes=("_dwi", "_t1"))['Patient_ID']
paired_patients_julian=pd.concat([first_dwi,first_t1])
paired_patients_julian=paired_patients_julian[paired_patients_julian["Patient_ID"].isin(paired_patients_julian_ids)]
unpaired_patients_julian=selection_julian_df[~selection_julian_df["Patient_ID"].isin(paired_patients_julian_ids)]

paired_patients_julian.sort_values("Patient_ID", inplace=True)
paired_patients_julian.info()

- martin

In [ ]:

with open(os.path.join(APP_DATA_PATH,'Martin_sequences.json')) as json_file:
    selection_martin_dict = json.load(json_file)


rows = []
for patid, selection_dict in selection_martin_dict.items():
    for modality, sequences in selection_dict.items():
        for sequence in sequences:
            study_id, sequence_id = sequence.split('/')
            rows.append({
                'Patient_ID': patid,
                'Study_ID': study_id,
                'Sequence_ID': sequence_id,
                'Modality': modality,
                'Labeller': 'Martin'  
            })
selection_martin_df = pd.DataFrame(rows).drop_duplicates()
len(selection_martin_df)

selection_martin_df['Sequence_path']=selection_martin_df.apply(lambda x: os.path.join(BERN_PATH,x['Patient_ID'],x['Study_ID'],x['Sequence_ID']),axis=1)

selection_martin_df['Date'] = selection_martin_df['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))

selection_martin_df


# Filter to keep only dwi and t1 modalities
df_filtered = selection_martin_df[selection_martin_df["Modality"].isin(["dwi", "t1"])]

first_dwi = df_filtered[df_filtered["Modality"] == "dwi"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()
first_t1 = df_filtered[df_filtered["Modality"] == "t1"].sort_values(["Patient_ID", "Date"]).groupby("Patient_ID").first().reset_index()

# only get thoes with exactly 2 mods
paired_patients_martin_ids = pd.merge(first_dwi, first_t1, on="Patient_ID", suffixes=("_dwi", "_t1"))['Patient_ID']
paired_patients_martin=pd.concat([first_dwi,first_t1])
paired_patients_martin=paired_patients_martin[paired_patients_martin["Patient_ID"].isin(paired_patients_martin_ids)]
unpaired_patients_martin=selection_martin_df[~selection_martin_df["Patient_ID"].isin(paired_patients_martin_ids)]

paired_patients_martin.sort_values("Patient_ID", inplace=True)
paired_patients_martin.info()


Unpaired

In [ ]:
unpaired_patients_hakim_=unpaired_patients_hakim.drop(index=[0,77,78,79]) # nothing to change
unpaired_patients_hakim_

In [ ]:
unpaired_patients_julian_=unpaired_patients_julian.drop(index=[3,13,14])  # these can be used since mostly there is a t2 or flair  13,14--> excluded because of the missing dwi
unpaired_patients_julian_.loc[202, "Modality"] = "dwi"
unpaired_patients_julian_.head(4)

In [ ]:
unpaired_patients_martin_=unpaired_patients_martin # nothing to change
unpaired_patients_martin_.head(4)  # these can be used since mostly there is a t2 or flair  

- MERGE overlap  # no overlap -> see set intersection

In [ ]:
set(paired_patients_hakim[paired_patients_hakim['Modality']=='dwi']['Patient_ID']).intersection(set(paired_patients_julian[paired_patients_julian['Modality']=='dwi']['Patient_ID'])).intersection(set(paired_patients_martin[paired_patients_martin['Modality']=='dwi']['Patient_ID'])).intersection(set(unpaired_patients_hakim_['Patient_ID'])).intersection(set(unpaired_patients_julian_['Patient_ID'])).intersection(set(unpaired_patients_martin_['Patient_ID']))

In [ ]:
selection_merged_df=pd.concat([paired_patients_hakim,paired_patients_julian,paired_patients_martin,unpaired_patients_hakim_,unpaired_patients_julian_,unpaired_patients_martin_])
selection_merged_df.sort_values(["Patient_ID", "Modality"])

- save mergefile

In [119]:
dwi_t1_df_merged = []  # 
for _, row in selection_merged_df.iterrows():
    for dcm_ending in os.listdir(row['Sequence_path']):        
        dwi_t1_df_merged.append({
            'Patient_ID': row['Patient_ID'],
            'Study_ID': row['Study_ID'],
            'Sequence_ID': row['Sequence_ID'],
            'File_Name': dcm_ending,
            'Full_Path': os.path.join(row['Sequence_path'], dcm_ending),
            'Modality': row['Modality'].upper()
        })

dwi_t1_df_merged = pd.DataFrame(dwi_t1_df_merged)
dwi_t1_df_merged['Date'] = dwi_t1_df_merged['Study_ID'].str[:8].apply(lambda x: datetime.strptime(x, '%Y%m%d').strftime('%Y-%m-%d'))
dwi_t1_df_merged['Sequence_ID_split'] = dwi_t1_df_merged['Sequence_ID'].apply(lambda x: split_at_anynumber_seq(x))

dwi_t1_df_merged["File_Number"] = dwi_t1_df_merged["File_Name"].str.extract(r'(\d+)').astype(int)
dwi_t1_df_merged.sort_values(["Patient_ID", "Modality", "File_Number"], inplace=True)
dwi_t1_df_merged.drop(columns=["File_Number"], inplace=True)
dwi_t1_df_merged.reset_index()
dwi_t1_df_merged.to_csv(os.path.join(ANALYSIS_PATH, 'dwi_t1_df_merged.csv'), index=False)